# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://images.datacamp.com/image/upload/v1676303379/Marketing/Blog/PySpark_RDD_Cheat_Sheet.pdf) is useful.  As is, the [RDD API reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.html).

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

In [287]:
# Take samples of the datasets
# sp_rddCitations = rddCitations.sample(False, 0.05,1)
# sp_rddPatents = rddPatents.sample(False, 0.05,1)

# sp_rddPatents.take(5)
#sp_rddCitations.take(5)

In [266]:
def patent_parser(x):
    line = x.split(",")
    patent = int(line[0])
    postate = line[1:]
    
    return (patent, postate)

def has_postate(x):
    patent = x[0]
    patent_info = x[1]
    if patent_info[4]=='""':
        return False
    else:
        return True

def patent_state_parser(x):
    line = x.split(",")
    patent = int(line[0])
    postate = line[5].replace('"','')
    
    return (patent, postate)

In [307]:
#patents = sp_rddPatents.map(lambda x: patent_parser(x))
header1 = rddPatents.first()
patents = rddPatents.filter(lambda line: line != header1)
patents_full = patents.map(lambda x: patent_parser(x))
patents_full = patents_full.filter(lambda x: has_postate(x))
#state_patents = sp_rddPatents.map(lambda x: patent_state_parser(x))
state_patents = patents.map(lambda x: patent_state_parser(x))
state_patents_filtered = state_patents.filter(lambda x: x[1]!='')

#patents_full.take(5)
state_patents_filtered.first()

(3070802, 'TX')

In [308]:
def citation_parser(x):
    line = x.split(",")
    citing = int(line[0])
    cited = int(line[1])
    
    return (citing,cited)

In [309]:
# Convert citation table into tuples
# citations = sp_rddCitations.map(lambda x: citation_parser(x))
header2 = rddCitations.first()
citations = rddCitations.filter(lambda line: line != header2)
citations = citations.map(lambda x: citation_parser(x))
citations.take(5)

[(3858241, 956203),
 (3858241, 1324234),
 (3858241, 3398406),
 (3858241, 3557384),
 (3858241, 3634889)]

In [310]:
# Join tables together using citing==patent comparison
citations1 = citations.join(state_patents_filtered)
citations1.take(5)

[(3879250, (1871492, 'PA')),
 (3879250, (2140692, 'PA')),
 (3879250, (3458193, 'PA')),
 (3879250, (3578526, 'PA')),
 (3879250, (3600257, 'PA'))]

In [313]:
# Swap citing and cited value positions for easy join
def swapper(x):
    citing = x[0]
    cited = x[1][0]
    postate = x[1][1]
    return (cited, (citing, postate))

swapped = citations1.map(lambda x: swapper(x))

# Join tables together using cited==patent comparison
citations2 = swapped.join(state_patents_filtered)
citations2.take(5)

[(5074326, ((5623968, 'OH'), 'CA')),
 (4804203, ((5999868, 'TX'), 'MI')),
 (4804203, ((5242190, 'MI'), 'MI')),
 (4804203, ((4921272, 'NC'), 'MI')),
 (4804203, ((4936425, 'NC'), 'MI'))]

In [314]:
# Reorganize RDD column order for easier understanding
def organizer(x):
    cited = x[0]
    citing = x[1][0][0]
    citing_postate = x[1][0][1]
    cited_postate = x[1][1]
    return (citing, citing_postate, cited, cited_postate)

# New order is: citing, citing_postate, cited, cited_postate
citations3 = citations2.map(lambda x: organizer(x))
citations3.take(5)

[(5202034, 'NH', 3317052, 'UT'),
 (3992298, 'UT', 3317052, 'UT'),
 (5527578, 'MN', 4504629, 'CT'),
 (5514730, 'MN', 4504629, 'CT'),
 (5907018, 'MN', 4504629, 'CT')]

In [316]:
# Determine if the citing state and cited state are the same for
# a Co-State Citation(csc)
def filter_csc(x):
    citing_postate = x[1]
    cited_postate = x[3]
    
    if citing_postate == cited_postate:
        return True
    else:
        return False
co_cited = citations3.filter(lambda x: filter_csc(x))
co_cited.take(5)

[(5290702, 'CA', 4992381, 'CA'),
 (5177221, 'CA', 4992381, 'CA'),
 (5371690, 'MA', 4972359, 'MA'),
 (5467446, 'MA', 4972359, 'MA'),
 (5801966, 'MA', 4972359, 'MA')]

In [320]:
# Do the basic map function discussed during week 1
def mapKeyToCounter(x):
    citing = x[0]
    return (citing,1)

count = co_cited.map(mapKeyToCounter).reduceByKey(lambda citing, cited: citing + cited)
# Sort the highest counted citing to the top
count = count.sortBy(lambda x: x[1], ascending=False)
count.take(10)

[(5959466, 125),
 (5983822, 103),
 (6008204, 100),
 (5952345, 98),
 (5958954, 96),
 (5998655, 96),
 (5936426, 94),
 (5925042, 90),
 (5913855, 90),
 (5951547, 90)]

In [321]:
# Join to table of all patent info parsed earlier
final_result = patents_full.join(count)
final_result = final_result.sortBy(lambda x: x[1][1], ascending=False)

In [323]:
final_result.take(5)

[(5959466,
  (['1999',
    '14515',
    '1997',
    '"US"',
    '"CA"',
    '5310',
    '2',
    '',
    '326',
    '4',
    '46',
    '159',
    '0',
    '1',
    '',
    '0.6186',
    '',
    '4.8868',
    '0.0455',
    '0.044',
    '',
    ''],
   125)),
 (5983822,
  (['1999',
    '14564',
    '1998',
    '"US"',
    '"TX"',
    '569900',
    '2',
    '',
    '114',
    '5',
    '55',
    '200',
    '0',
    '0.995',
    '',
    '0.7201',
    '',
    '12.45',
    '0',
    '0',
    '',
    ''],
   103)),
 (6008204,
  (['1999',
    '14606',
    '1998',
    '"US"',
    '"CA"',
    '749584',
    '2',
    '',
    '514',
    '3',
    '31',
    '121',
    '0',
    '1',
    '',
    '0.7415',
    '',
    '5',
    '0.0085',
    '0.0083',
    '',
    ''],
   100)),
 (5952345,
  (['1999',
    '14501',
    '1997',
    '"US"',
    '"CA"',
    '749584',
    '2',
    '',
    '514',
    '3',
    '31',
    '118',
    '0',
    '1',
    '',
    '0.7442',
    '',
    '5.1102',
    '0',
    '0',
    '',


In [324]:
for patent,info in final_result.take(5):
    details = info[0]
    count = info[1]
    print(f"({patent}, {details}, {count})")

(5959466, ['1999', '14515', '1997', '"US"', '"CA"', '5310', '2', '', '326', '4', '46', '159', '0', '1', '', '0.6186', '', '4.8868', '0.0455', '0.044', '', ''], 125)
(5983822, ['1999', '14564', '1998', '"US"', '"TX"', '569900', '2', '', '114', '5', '55', '200', '0', '0.995', '', '0.7201', '', '12.45', '0', '0', '', ''], 103)
(6008204, ['1999', '14606', '1998', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '121', '0', '1', '', '0.7415', '', '5', '0.0085', '0.0083', '', ''], 100)
(5952345, ['1999', '14501', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '118', '0', '1', '', '0.7442', '', '5.1102', '0', '0', '', ''], 98)
(5958954, ['1999', '14515', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '116', '0', '1', '', '0.7397', '', '5.181', '0', '0', '', ''], 96)
